In [ ]:
import os

os.environ["GROQ_API_KEY"] = ""
os.environ["ACTIVELOOP_TOKEN"] = ""

print("API keys set successfully")

API keys set successfully


In [2]:
from langchain_community.document_loaders import SeleniumURLLoader
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Automatic ChromeDriver setup
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

urls = [
    "https://beebom.com/how-check-disk-usage-linux/",
    "https://beebom.com/how-delete-spotify-account/"
]

loader = SeleniumURLLoader(urls=urls)
documents = loader.load()

print("Documents Loaded:", len(documents))

Documents Loaded: 2


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)

print("Total Chunks Created:", len(docs))

Total Chunks Created: 61


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

print("Embedding Model Loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Model Loaded


In [5]:
from langchain_community.vectorstores import DeepLake

dataset_path = "hub://chalamalasettyakashmadhukar/groq_rag_demo"

vectorstore = DeepLake.from_documents(
    docs,
    embedding=embedding_model,
    dataset_path=dataset_path,
    overwrite=True
)

print("Deep Lake Vector Store Created Successfully")

C:\Users\Chala\OneDrive\Desktop\CustomerSupport_RAG\rag_env\lib\site-packages\deeplake\util\check_latest_version.py:32: UserWarning: A newer version of deeplake (4.5.2) is available. It's recommended that you update to the latest version using `pip install -U deeplake`.
  warnings.warn(


Your Deep Lake dataset has been successfully created!


Creating 61 embeddings in 1 batches of size 61:: 100%|███████████████████████████████████| 1/1 [00:35<00:00, 35.24s/it]

Dataset(path='hub://chalamalasettyakashmadhukar/groq_rag_demo', tensors=['text', 'metadata', 'embedding', 'id'])

  tensor      htype      shape     dtype  compression
  -------    -------    -------   -------  ------- 
   text       text      (61, 1)     str     None   
 metadata     json      (61, 1)     str     None   
 embedding  embedding  (61, 768)  float32   None   
    id        text      (61, 1)     str     None   
Deep Lake Vector Store Created Successfully


In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Retriever Ready")

Retriever Ready


In [7]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

print("Groq Ready")

Groq Ready


In [8]:
def generate_answer(query):
    retrieved_docs = retriever.get_relevant_documents(query)
    
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    prompt = f"""
You are a professional customer support assistant.

Answer ONLY using the context below.

If the answer is not found in the context, say:
"I cannot find this information in the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""
    
    response = llm.invoke(prompt)
    return response.content

In [9]:
query = "How to check disk usage in Linux?"
answer = generate_answer(query)

print("Answer:\n")
print(answer)

C:\Users\Chala\OneDrive\Desktop\CustomerSupport_RAG\rag_env\lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


Answer:

There are several ways to check disk usage in Linux. 

1. Using the df command: You can use the df command to check the current disk usage and the available disk space in Linux. The syntax for the df command is df <options> <file_system>.

2. Using the du command: The df command only shows the disk usage for the entire file system and not for individual files and directories. To view the disk usage for individual files and directories, use the du command. The syntax to use the du command is du <option> <file>.

3. Using GUI tools: If you find the command line output hard to understand, you can use GUI tools like GDU Disk Usage Analyzer and the Gnome Disks Tool. These tools can be easily installed using the following commands: 
- sudo snap install gdu-disk-usage-analyzer
- Installing disk-utility tool:
